# Index · Unit 1 — Reasoning & control (the "brain")

For every component in Unit 1 of `index.md`, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs with a scripted
  *fake model* (no API key needed) so you can see every moving part.
- **SOTA** — how it's actually done today, using the OpenAI SDK / native model
  features. Needs `OPENAI_API_KEY`.

Components: **1** model choice · **2** system prompt · **3** the agent loop ·
**4** reasoning patterns (ReAct / CoT / Reflexion) · **5** planning &
decomposition · **6** routing · **7** termination conditions.

## Setup

In [ ]:
import os, re, json

MODEL = "gpt-4o-mini"   # small, tool-capable model for the SOTA cells

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    """Thin wrapper over the OpenAI SDK. Client built lazily so the
    from-scratch cells run without a key."""
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs
    ).choices[0].message

## 1. LLM / model choice

A model call is just **text in → text out**. "Choosing a model" is a *selector*:
map a task to the right model by capability vs cost/latency.

**From scratch:** a model registry + a rule that picks the cheapest model whose
reasoning power clears the task's requirement.

In [ ]:
# From scratch — rule-based selection over a registry.
MODELS = {
    "small": {"cost": 1,  "ctx": 128_000, "reasoning": 2},
    "mid":   {"cost": 3,  "ctx": 200_000, "reasoning": 4},
    "large": {"cost": 15, "ctx": 200_000, "reasoning": 5},
}

def choose_model(required_reasoning, min_ctx=0):
    ok = {n: m for n, m in MODELS.items()
          if m["reasoning"] >= required_reasoning and m["ctx"] >= min_ctx}
    if not ok:
        raise ValueError("no model meets the requirement")
    return min(ok, key=lambda n: MODELS[n]["cost"])   # cheapest that qualifies

print("easy task   ->", choose_model(required_reasoning=2))
print("hard task   ->", choose_model(required_reasoning=5))
print("big context ->", choose_model(required_reasoning=2, min_ctx=150_000))

In [ ]:
# SOTA — let a cheap model classify difficulty, then reuse the selector above.
# (In production you'd also query OpenAI().models.list() to discover models.)
def llm_route(task):
    m = chat([
        {"role": "system", "content": "Rate this task's reasoning difficulty 1-5. Reply with ONLY the digit."},
        {"role": "user", "content": task},
    ])
    difficulty = int(re.search(r"[1-5]", m.content).group())
    return difficulty, choose_model(difficulty)

for t in ["Add 2 and 3", "Prove the infinitude of primes with a novel argument"]:
    print(t, "->", llm_route(t))

## 2. System prompt / instructions

A base (completion) model has no "system" role — instructions are just
**privileged text placed first**. Chat models formalize that as a `system`-role
message. It steers behavior because the model was trained to prioritize it.

**From scratch:** build the message list by hand and flatten it to exactly what
a base model "sees"; a toy model shows the instructions changing the output.

In [ ]:
# From scratch — a system prompt is just text with priority-by-convention.
def render(messages):
    return "\n\n".join(f"[{m['role'].upper()}]\n{m['content']}" for m in messages)

def toy_model(messages):
    sys = next((m["content"] for m in messages if m["role"] == "system"), "").lower()
    ans = "Paris is the capital of France."
    if "one word" in sys: ans = "Paris"
    if "uppercase" in sys: ans = ans.upper()
    return ans

for instruction in ["You are helpful.", "Answer in one word.", "Answer in one word. Use UPPERCASE."]:
    msgs = [{"role": "system", "content": instruction},
            {"role": "user", "content": "What is the capital of France?"}]
    print(repr(instruction), "->", toy_model(msgs))

print("\n--- what a base model actually sees ---")
print(render(msgs))

In [ ]:
# SOTA — the system role on a real model; same question, different instructions.
for instruction in ["You are terse. Answer in one word.",
                    "You are a cheerful tour guide. Answer in 2 enthusiastic sentences."]:
    m = chat([{"role": "system", "content": instruction},
              {"role": "user", "content": "What is the capital of France?"}])
    print(repr(instruction), "->", m.content)

## 3. The agent loop

The control cycle: **reason → act → observe → repeat**. (You built the full
version in Phase 1; here it is condensed.)

**From scratch:** a fake model emits an `ACT:` line; the harness runs the tool,
feeds back an `OBS:` line, and loops until the model answers.

In [ ]:
# From scratch — hand-rolled reason/act/observe loop.
def add(a, b): return a + b
TOOLS = {"add": add}

def fake(messages):
    observed = [m for m in messages if m["content"].startswith("OBS:")]
    if not observed:
        return 'THOUGHT: I should add.\nACT: {"tool": "add", "args": {"a": 2, "b": 3}}'
    result = observed[-1]["content"].split("OBS:")[-1].strip()
    return f"ANSWER: the result is {result}"

def agent_loop(task, model, max_steps=5):
    msgs = [{"role": "user", "content": task}]
    for _ in range(max_steps):
        out = model(msgs)
        print("model:", out.replace("\n", " | "))
        msgs.append({"role": "assistant", "content": out})
        m = re.search(r'ACT:\s*(\{.*\})', out)
        if not m:
            return out                              # no action -> final answer
        call = json.loads(m.group(1))
        obs = TOOLS[call["tool"]](**call["args"])
        msgs.append({"role": "user", "content": f"OBS: {obs}"})
    return "(stopped: max steps)"

print("\nfinal:", agent_loop("Add 2 and 3.", fake))

In [ ]:
# SOTA — the same loop via native tool calling (frameworks: LangGraph, OpenAI Agents SDK).
def sota_agent(task, max_steps=5):
    tools = [{"type": "function", "function": {
        "name": "add", "description": "Add two numbers",
        "parameters": {"type": "object",
                       "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                       "required": ["a", "b"]}}}]
    msgs = [{"role": "user", "content": task}]
    for _ in range(max_steps):
        m = chat(msgs, tools=tools, tool_choice="auto")
        msgs.append(m)
        if not m.tool_calls:
            return m.content
        for tc in m.tool_calls:
            res = add(**json.loads(tc.function.arguments))
            print(f"  🔧 {tc.function.name} -> {res}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(res)})
    return "(stopped: max steps)"

print(sota_agent("What is 2 + 3?"))

## 4. Reasoning patterns

- **ReAct** = the loop in §3 with explicit *Thought → Action → Observation*.
- **Chain-of-Thought (CoT)** = think step by step *before* answering.
- **Reflexion** = generate → self-critique → revise.
- *(Also: Tree-of-Thoughts — branch & score; Plan-and-Execute — see §5;
  ReWOO — plan all tool calls up front, then execute.)*

Below: CoT and Reflexion, from scratch then SOTA.

In [ ]:
# From scratch — CoT is a prompt convention: ask for steps, then parse the final line.
COT_INSTRUCTION = "Think step by step, then end with a line 'FINAL: <answer>'."

def fake_cot(question):
    return "Step 1: 3 boxes.\nStep 2: 4 apples each.\nStep 3: 3*4 = 12.\nFINAL: 12"

def extract_final(text):
    return text.split("FINAL:")[-1].strip()

raw = fake_cot("3 boxes, 4 apples each — how many apples?")
print(raw)
print("parsed answer ->", extract_final(raw))

In [ ]:
# SOTA — CoT via the system prompt (modern "reasoning" models do this internally).
m = chat([{"role": "system", "content": COT_INSTRUCTION},
          {"role": "user", "content": "A shop has 3 boxes with 4 apples each. How many apples total?"}])
print(m.content)
print("\nparsed answer ->", extract_final(m.content))

In [ ]:
# From scratch — Reflexion: draft -> critique -> revise, all hand-wired.
def draft(q):     return "The answer is 40."
def critique(a):  return "correct" if "42" in a else "Wrong: recompute, it should be 42."
def revise(a, fb): return "The answer is 42."

def reflexion(q, rounds=3):
    ans = draft(q)
    for i in range(rounds):
        fb = critique(ans)
        print(f"round {i}: {ans!r}  <- {fb!r}")
        if "correct" in fb.lower():
            break
        ans = revise(ans, fb)
    return ans

print("final:", reflexion("6 * 7 ?"))

In [ ]:
# SOTA — Reflexion with a real model: one call drafts, a second critiques, a third revises.
def reflexion_sota(question, rounds=2):
    ans = chat([{"role": "user", "content": question}]).content
    for _ in range(rounds):
        fb = chat([{"role": "system", "content": "Critique the answer. If fully correct reply exactly 'CORRECT', else explain the fix."},
                   {"role": "user", "content": f"Q: {question}\nA: {ans}"}]).content
        if fb.strip().upper().startswith("CORRECT"):
            break
        ans = chat([{"role": "user", "content": f"Q: {question}\nYour previous answer: {ans}\nCritique: {fb}\nGive the corrected final answer."}]).content
    return ans

print(reflexion_sota("What is 17 * 23? Answer with just the number."))

## 5. Planning & task decomposition

Break a goal into ordered subtasks, then execute them.

**From scratch:** parse a numbered plan from the model's text into a list.
**SOTA:** get a *typed* plan directly via structured outputs (JSON schema) — no
brittle text parsing.

In [ ]:
# From scratch — parse a numbered plan, then run each step.
def fake_planner(goal):
    return "1. Gather the sales data\n2. Compute monthly totals\n3. Write the summary"

def parse_plan(text):
    return [re.sub(r'^\s*\d+[\.\)]\s*', '', line).strip()
            for line in text.splitlines() if line.strip()]

plan = parse_plan(fake_planner("Write a sales report"))
for i, step in enumerate(plan, 1):
    print(f"executing step {i}: {step}")

In [ ]:
# SOTA — structured outputs give a typed plan; no regex.
plan_schema = {
    "type": "object",
    "properties": {"steps": {"type": "array", "items": {"type": "string"}}},
    "required": ["steps"],
    "additionalProperties": False,
}
m = chat(
    [{"role": "system", "content": "Break the goal into 3-6 concrete ordered steps."},
     {"role": "user", "content": "Goal: launch a personal blog"}],
    response_format={"type": "json_schema",
                     "json_schema": {"name": "plan", "schema": plan_schema, "strict": True}},
)
for i, step in enumerate(json.loads(m.content)["steps"], 1):
    print(f"{i}. {step}")

## 6. Routing

Send each request to the right handler/agent/model.

**From scratch:** deterministic keyword rules.
**SOTA:** an LLM classifier constrained to a fixed set of routes (structured output).

In [ ]:
# From scratch — keyword router.
def route(query):
    q = query.lower()
    if any(w in q for w in ["add", "sum", "multiply", "calculate", "times"]):
        return "math_agent"
    if any(w in q for w in ["weather", "forecast", "temperature"]):
        return "weather_agent"
    return "general_agent"

for q in ["What's 12 times 8?", "Weather in Paris tomorrow?", "Tell me a joke"]:
    print(f"{q!r:40} -> {route(q)}")

In [ ]:
# SOTA — LLM classifier locked to an enum of routes.
ROUTES = ["math_agent", "weather_agent", "general_agent"]
route_schema = {
    "type": "object",
    "properties": {"route": {"type": "string", "enum": ROUTES}},
    "required": ["route"],
    "additionalProperties": False,
}
def llm_router(query):
    m = chat(
        [{"role": "system", "content": "Pick the best agent to handle the user's request."},
         {"role": "user", "content": query}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "route", "schema": route_schema, "strict": True}},
    )
    return json.loads(m.content)["route"]

for q in ["What's 12 times 8?", "Weather in Paris tomorrow?", "Tell me a joke"]:
    print(f"{q!r:40} -> {llm_router(q)}")

## 7. Termination / stopping conditions

An agent loop must know when to stop. Common guards:
**max steps**, **goal met**, **no action / done signal**, and **loop detection**
(repeating the same action).

**From scratch:** a loop that surfaces exactly which condition fired.

In [ ]:
# From scratch — explicit termination guards.
def run(policy, goal_met, max_steps=5):
    history = []
    for step in range(max_steps):
        action = policy(history)
        if action == "DONE":
            return f"stopped: model signalled done at step {step}"
        if action in history:
            return f"stopped: loop detected ({action!r} repeated) at step {step}"
        history.append(action)
        if goal_met(history):
            return f"stopped: goal met at step {step} (history={history})"
    return f"stopped: hit max_steps={max_steps} (history={history})"

# each toy policy triggers a different stop condition
print(run(lambda h: "DONE", goal_met=lambda h: False))
print(run(lambda h: "a", goal_met=lambda h: False))                 # loop: 'a' repeats
print(run(lambda h: f"step{len(h)}", goal_met=lambda h: len(h) == 2))
print(run(lambda h: f"step{len(h)}", goal_met=lambda h: False))     # runs out of steps

In [ ]:
# SOTA — same guards around a real loop, plus a 'finish' tool the model calls to
# signal completion (cleaner than parsing text for "done"). Needs a key.
def agent_with_finish(task, max_steps=6):
    tools = [
        {"type": "function", "function": {"name": "add",
            "parameters": {"type": "object",
                           "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                           "required": ["a", "b"]}, "description": "Add two numbers"}},
        {"type": "function", "function": {"name": "finish",
            "parameters": {"type": "object",
                           "properties": {"answer": {"type": "string"}},
                           "required": ["answer"]}, "description": "Call when the task is complete"}},
    ]
    msgs = [{"role": "user", "content": task}]
    for step in range(max_steps):                       # <-- max-steps guard
        m = chat(msgs, tools=tools, tool_choice="auto")
        msgs.append(m)
        if not m.tool_calls:
            return f"stopped (end_turn): {m.content}"
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            if tc.function.name == "finish":            # <-- explicit done signal
                return f"stopped (finish) at step {step}: {args['answer']}"
            res = args["a"] + args["b"]
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(res)})
    return "stopped: max_steps"

print(agent_with_finish("Add 2 and 3, then report the result and finish."))

## Recap

Every Unit 1 component split cleanly into **the mechanism** (a prompt / a loop /
a selector / a guard you can write in a few lines) and **the SOTA packaging**
(native tool calling, structured outputs, reasoning models, agent frameworks).

The from-scratch versions are what frameworks like LangGraph and the OpenAI
Agents SDK implement for you — now you know what's under the hood.

Next units in `index.md` (tools ✅, MCP ✅ already built in Phase 1; memory,
RAG, multi-agent, guardrails, eval, …) can follow this same two-cell pattern.